In [1]:
!apt-get update -qq
!apt-get install -y -qq zstd

!pip install -q -U \
    llama-index \
    llama-index-llms-ollama \
    llama-index-embeddings-ollama \
    llama-index-vector-stores-chroma \
    chromadb

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("✓ Ollama server started")

✓ Ollama server started


In [4]:
!ollama --version
!ollama list

ollama version is 0.34.1
NAME    ID    SIZE    MODIFIED 


In [5]:
!ollama pull nomic-embed-text

In [6]:
import os
import time
import shutil
import pandas as pd

from pathlib import Path

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    Settings
)

from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

print("✓ LlamaIndex imports successful")

✓ LlamaIndex imports successful


In [7]:
Settings.llm = Ollama(
    model="llama3.2:3b",
    request_timeout=120.0
)

Settings.embed_model = OllamaEmbedding(
    model_name="nomic-embed-text",
    base_url="http://localhost:11434"
)

print("✓ Ollama LLM configured")
print("✓ Ollama embedding model configured")

✓ Ollama LLM configured
✓ Ollama embedding model configured


In [8]:
data_dir = Path("documents")

data_dir.mkdir(exist_ok=True)

sample_documents = {
    "ai.txt": """
Artificial Intelligence is a field of computer science that focuses on
creating systems capable of performing tasks that normally require human
intelligence. These tasks include learning, reasoning, perception and
decision making.
""",

    "machine_learning.txt": """
Machine Learning is a subset of Artificial Intelligence. It allows
computers to learn patterns from data and make predictions or decisions
without being explicitly programmed for every task.
""",

    "deep_learning.txt": """
Deep Learning is a branch of Machine Learning that uses neural networks
with multiple layers. Deep learning is widely used in computer vision,
natural language processing and speech recognition.
""",

    "rag.txt": """
Retrieval Augmented Generation, or RAG, combines information retrieval
with language generation. Relevant documents are retrieved first and
then provided to a language model to generate a grounded answer.
""",

    "vector_database.txt": """
A vector database stores numerical representations called embeddings.
These embeddings allow systems to search for documents based on semantic
similarity rather than only matching exact keywords.
"""
}

for filename, content in sample_documents.items():

    with open(data_dir / filename, "w") as f:
        f.write(content)

print("✓ Text files created")

for file in data_dir.iterdir():
    print("-", file.name)

✓ Text files created
- rag.txt
- vector_database.txt
- ai.txt
- machine_learning.txt
- deep_learning.txt


In [9]:
documents = SimpleDirectoryReader(
    input_dir=str(data_dir)
).load_data()

print("✓ Documents loaded:", len(documents))

for doc in documents:
    print("-", doc.metadata.get("file_name"))

✓ Documents loaded: 5
- ai.txt
- deep_learning.txt
- machine_learning.txt
- rag.txt
- vector_database.txt


In [10]:
start_time = time.time()

index = VectorStoreIndex.from_documents(
    documents
)

indexing_latency = time.time() - start_time

print("✓ VectorStoreIndex created")
print("Indexing time:", round(indexing_latency, 3), "seconds")

✓ VectorStoreIndex created
Indexing time: 2.181 seconds


In [12]:
!ollama list

NAME                       ID              SIZE      MODIFIED      
nomic-embed-text:latest    0a109f422b47    274 MB    3 minutes ago    


In [13]:
!ollama pull llama3.2:3b

In [14]:
!ollama list

NAME                       ID              SIZE      MODIFIED      
llama3.2:3b                a80c4f17acd5    2.0 GB    4 seconds ago    
nomic-embed-text:latest    0a109f422b47    274 MB    5 minutes ago    


In [15]:
from llama_index.llms.ollama import Ollama

Settings.llm = Ollama(
    model="llama3.2:3b",
    request_timeout=120.0,
    context_window=4096
)

print("✓ LlamaIndex LLM configured")

✓ LlamaIndex LLM configured


In [16]:
query_engine = index.as_query_engine(
    similarity_top_k=3
)

print("✓ QueryEngine created")

✓ QueryEngine created


In [17]:
queries = [
    "What is Artificial Intelligence?",
    "What is Machine Learning?",
    "What is Deep Learning?",
    "What are neural networks used for?",
    "What is Retrieval Augmented Generation?",
    "How does RAG work?",
    "What is a vector database?",
    "What are embeddings?",
    "How is Deep Learning related to Machine Learning?",
    "Why are vector databases useful for RAG?"
]

print("Total queries:", len(queries))

Total queries: 10


In [18]:
results = []

for i, query in enumerate(queries, start=1):

    start = time.time()

    response = query_engine.query(query)

    latency = time.time() - start

    results.append({
        "Query": query,
        "Answer": str(response),
        "Latency (sec)": round(latency, 3)
    })

    print("=" * 80)
    print(f"QUERY {i}")
    print(query)

    print("\nANSWER:")
    print(response)

    print("\nLatency:",
          round(latency, 3),
          "seconds")

QUERY 1
What is Artificial Intelligence?

ANSWER:
Artificial Intelligence is a field of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence, such as learning, reasoning, perception and decision making.

Latency: 53.217 seconds
QUERY 2
What is Machine Learning?

ANSWER:
Computers can learn patterns from data and make predictions or decisions without being explicitly programmed for every task.

Latency: 26.599 seconds
QUERY 3
What is Deep Learning?

ANSWER:
Deep Learning is a branch of Machine Learning that employs neural networks with multiple layers to analyze and interpret complex data, enabling accurate predictions and decisions in various fields such as computer vision, natural language processing, and speech recognition.

Latency: 33.868 seconds
QUERY 4
What are neural networks used for?

ANSWER:
Neural networks are used for a wide range of tasks that require pattern recognition and complex decision making.

Latency

In [19]:
for i, query in enumerate(queries, start=1):

    response = query_engine.query(query)

    print("=" * 80)
    print(f"QUERY {i}: {query}")

    print("\nANSWER:")
    print(response)

    print("\nSOURCE DOCUMENTS:")

    for source_node in response.source_nodes:

        print(
            "-",
            source_node.metadata.get(
                "file_name",
                "Unknown"
            )
        )

QUERY 1: What is Artificial Intelligence?

ANSWER:
Artificial Intelligence is a field of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence, such as learning, reasoning, perception, and decision making.

SOURCE DOCUMENTS:
- ai.txt
- machine_learning.txt
- deep_learning.txt
QUERY 2: What is Machine Learning?

ANSWER:
Machine Learning is a subset of Artificial Intelligence that enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task.

SOURCE DOCUMENTS:
- machine_learning.txt
- deep_learning.txt
- ai.txt
QUERY 3: What is Deep Learning?

ANSWER:
Deep Learning is a branch of Machine Learning that uses neural networks with multiple layers.

SOURCE DOCUMENTS:
- deep_learning.txt
- machine_learning.txt
- ai.txt
QUERY 4: What are neural networks used for?

ANSWER:
Neural networks are primarily used in tasks that require complex pattern recognition, such 

In [20]:
import chromadb

from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

chroma_collection = chroma_client.get_or_create_collection(
    name="w7d3_documents"
)

vector_store = ChromaVectorStore(
    chroma_collection=chroma_collection
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

print("✓ ChromaDB vector store created")

✓ ChromaDB vector store created


In [21]:
start_time = time.time()

chroma_index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context
)

chroma_indexing_latency = time.time() - start_time

print("✓ LlamaIndex connected to ChromaDB")
print(
    "ChromaDB indexing time:",
    round(chroma_indexing_latency, 3),
    "seconds"
)

✓ LlamaIndex connected to ChromaDB
ChromaDB indexing time: 1.493 seconds


In [22]:
chroma_query_engine = chroma_index.as_query_engine(
    similarity_top_k=3
)

print("✓ ChromaDB QueryEngine created")

✓ ChromaDB QueryEngine created


In [23]:
chroma_results = []

for i, query in enumerate(queries, start=1):

    start = time.time()

    response = chroma_query_engine.query(query)

    latency = time.time() - start

    chroma_results.append({
        "Query": query,
        "Answer": str(response),
        "Latency (sec)": round(latency, 3)
    })

    print("=" * 80)
    print(f"QUERY {i}")
    print(query)

    print("\nANSWER:")
    print(response)

    print("\nLatency:",
          round(latency, 3),
          "seconds")

QUERY 1
What is Artificial Intelligence?

ANSWER:
A field of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence, such as learning, reasoning, perception, and decision making.

Latency: 14.03 seconds
QUERY 2
What is Machine Learning?

ANSWER:
Computers can learn patterns from data and make predictions or decisions without being explicitly programmed for every task.

Latency: 8.293 seconds
QUERY 3
What is Deep Learning?

ANSWER:
Deep Learning is a specialized form of Machine Learning that uses complex neural networks with many layers to analyze data and make accurate predictions or decisions.

Latency: 11.52 seconds
QUERY 4
What are neural networks used for?

ANSWER:
Neural networks are widely used in tasks that require complex pattern recognition and decision making. They are particularly useful in applications where data is abundant and dynamic, allowing them to learn and adapt over time.

Latency: 16.367 seconds
QUERY

In [24]:
comparison_df = pd.DataFrame({
    "Query": queries,

    "LlamaIndex Latency (sec)": [
        item["Latency (sec)"]
        for item in results
    ],

    "ChromaDB Latency (sec)": [
        item["Latency (sec)"]
        for item in chroma_results
    ]
})

comparison_df["Difference (sec)"] = (
    comparison_df["ChromaDB Latency (sec)"]
    - comparison_df["LlamaIndex Latency (sec)"]
)

display(comparison_df)

,Query,LlamaIndex Latency (sec),ChromaDB Latency (sec),Difference (sec)
0,What is Artificial Intelligence?,53.217,14.030,-39.187
1,What is Machine Learning?,26.599,8.293,-18.306
2,What is Deep Learning?,33.868,11.520,-22.348
3,What are neural networks used for?,8.329,16.367,8.038
4,What is Retrieval Augmented Generation?,23.387,41.920,18.533
5,How does RAG work?,26.246,43.525,17.279
6,What is a vector database?,29.985,19.168,-10.817
7,What are embeddings?,20.961,8.046,-12.915
8,How is Deep Learning related to Machine Learning?,16.531,16.118,-0.413
9,Why are vector databases useful for RAG?,10.388,15.761,5.373


In [25]:
llama_avg = comparison_df[
    "LlamaIndex Latency (sec)"
].mean()

chroma_avg = comparison_df[
    "ChromaDB Latency (sec)"
].mean()

print(
    "Average LlamaIndex latency:",
    round(llama_avg, 3),
    "seconds"
)

print(
    "Average ChromaDB latency:",
    round(chroma_avg, 3),
    "seconds"
)

Average LlamaIndex latency: 24.951 seconds
Average ChromaDB latency: 19.475 seconds


In [26]:
comparison_df.to_csv(
    "w7d3_llamaindex_chromadb_latency.csv",
    index=False
)

pd.DataFrame(results).to_csv(
    "w7d3_llamaindex_results.csv",
    index=False
)

pd.DataFrame(chroma_results).to_csv(
    "w7d3_chromadb_results.csv",
    index=False
)

print("✓ Results saved successfully")

✓ Results saved successfully
